# Shoof fill model (step 3)
Teaches Shoof to paint what was behind an object or text after it is lifted out — sharp, not a blurry smear. It learns from the same data as step 2: every design and photo poster is built from exact layers, so the true picture behind each layer is known.

**Inputs:** + Add Input → Your Work → the step 1 notebook (designs) AND the step 1b notebook (photo posters).
**Settings:** Accelerator **GPU T4 x2**, Internet **On**. Then **Save Version → Save & Run All**. It stops by itself after about 10½ hours (Kaggle's limit is 12).
**Download when done:** `shoof-fill-fp16.onnx`, `fill-preview.png`, `fill-report.json`.

In [ ]:
!git clone -q --depth 1 https://github.com/shahrosekhalid96-debug/shoof /kaggle/temp/shoof
!pip install -q onnx onnxconverter-common
import glob, os, tarfile, shutil
print('Inputs attached:', os.listdir('/kaggle/input') if os.path.isdir('/kaggle/input') else 'none')
os.makedirs('/kaggle/temp/ld', exist_ok=True)
# Step 1 output: ld-*.tar files (Kaggle sometimes unpacks them into folders instead)
tars = sorted(glob.glob('/kaggle/input/**/ld-*.tar', recursive=True) + glob.glob('/kaggle/input/**/lp-*.tar', recursive=True))
for t in tars:
    tarfile.open(t).extractall('/kaggle/temp/ld')
for m in glob.glob('/kaggle/input/**/meta.json', recursive=True):
    d = os.path.dirname(m); dst = os.path.join('/kaggle/temp/ld', os.path.basename(d))
    if not os.path.exists(dst): os.symlink(d, dst)
n = len(os.listdir('/kaggle/temp/ld'))
print(len(tars), 'tar files,', n, 'designs ready (', sum(1 for d in os.listdir('/kaggle/temp/ld') if d.isdigit() and int(d) >= 1000000), 'photo posters )')
assert n, ('No training data found. Check: 1) the step 1 notebook finished (green tick) and its Output tab shows ld-00.tar...; '
           '2) it is added here: right panel -> Add Input -> Your Work -> step 1 notebook.')

In [ ]:
!cd /kaggle/temp/shoof && python tools/layerdata/train_fill.py --data /kaggle/temp/ld --out /kaggle/working --epochs 12 --batch 12 --size 512 --workers 4 --hours 10.5
from IPython.display import Image as I, display; import json
display(I('/kaggle/working/fill-preview.png')); print(json.load(open('/kaggle/working/fill-report.json')))